# OPSD-2 — **d-OPSD** (future-token teacher) · SDAR-4B-b32 · local Jupyter

Replication of the d-OPSD formulation. Teacher and student are the same frozen base model with
**the same prompt** and **the same denoising schedule** — the teacher's only advantage is that a
few tokens from the *future* of the student's own reveal order are already filled in when it
scores the current positions.

Concretely, at teacher sub-step `s`:

* revealed as usual: the positions the student had revealed before sub-step `s`
* **plus** `cfg.future_reveal_k` positions the student will only reveal *later*, filled with the
  tokens the student actually committed

so the teacher's conditional is strictly better-informed than the student's at the same point,
with no reference solution and no extra sub-steps. Loss is clipped KL(teacher‖student) on the
positions revealed at that sub-step; prefixes advance with the student's tokens (on-policy).

**Contrast with OPSD-1** (the other notebook): there the teacher gets a privileged prompt
containing a verified reference solution *and* a 4× finer schedule. Here it gets neither — only
lookahead. Running both on the same benchmark is the comparison the paper needs.

---

## Environment: this notebook is for a **local Jupyter** install, not Colab

* No `google.colab` import, no Drive mount. All state goes to `cfg.drive_root`, a **local
  directory** (default `./opsd2_run`), created on first run.
* §2's torchao-removal / transformers-pin dance is **kept** — `modeling_sdar.py` needs
  transformers 4.5x, and the torchao uninstall must happen before any transformers import. On a
  clean local env with the right version already installed it is a harmless no-op; on a dirty
  one it saves you. **If it changes the transformers version, restart the kernel and re-run from
  the top.**
* Requires a CUDA GPU with roughly 40GB+ for 4B + LoRA at `max_gen_tokens=1536`. Lower
  `max_gen_tokens`, or set `cfg.load_in_4bit=True`, if VRAM is tighter.

## Performance repair carried over

The teacher's sub-step states are **batched** (`cfg.teacher_batch`) rather than run
sequentially — possible because once the student commits a block, every teacher input state is
known in advance. Here it matters less than in OPSD-1 (the teacher runs at the student's rate,
not 4× finer) but it is free. Per-problem `student / teacher / backward` timings print in §11.

## 1 · GPU check & local working directory

In [ ]:
import subprocess, torch, platform
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi not found — is this a GPU machine?")
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| Python:", platform.python_version())
assert torch.cuda.is_available(), "No GPU visible to torch — check drivers / CUDA install."
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")
if p.total_memory/1e9 < 38:
    print("⚠️ under ~40GB: lower cfg.max_gen_tokens, or set cfg.load_in_4bit=True in §3.")


In [ ]:
# Local run — no Drive. All checkpoints/logs go under cfg.drive_root (set in §3).
import os
WORKDIR = os.path.abspath("./opsd2_run")
os.makedirs(WORKDIR, exist_ok=True)
print("working directory:", WORKDIR)
print("(cfg.drive_root in §3 points here; change it there if you want another location)")


## 2 · Dependencies — pin transformers, remove torchao FIRST
`modeling_sdar.py` needs transformers 4.5x (Colab ships 5.x). We clone the repo, read its pin,
install it. **Critical ordering:** torchao is uninstalled *before* transformers is imported —
transformers caches `is_torchao_available()` at import time and then does a deferred torchao
import when loading the model, so removing it afterward is too late. **If the transformers
version changes, restart the runtime and re-run from the top.**

In [ ]:
import os, re, subprocess, sys

# 0) remove torchao BEFORE any transformers import (stale Colab build breaks peft + transformers)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
for _n in [n for n in list(sys.modules) if n == "torchao" or n.startswith("torchao.")]:
    del sys.modules[_n]

if not os.path.isdir('/content/dLLM-RL'):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/Gen-Verse/dLLM-RL","/content/dLLM-RL"], check=True)

_FALLBACK = "transformers==4.51.3"
_spec = _FALLBACK
_req = "/content/dLLM-RL/requirements.txt"
if os.path.isfile(_req):
    txt = open(_req).read()
    m = re.search(r"^\s*transformers(\[[^\]]*\])?\s*([=<>!~].*?)\s*(?:#.*)?$", txt, re.M)
    if m and m.group(2): _spec = "transformers" + (m.group(1) or "") + m.group(2).strip()
print("Pinning:", _spec, "(from requirements.txt)" if _spec != _FALLBACK else "(fallback)")

!pip -q install "{_spec}" "accelerate>=0.33" "peft>=0.12" "datasets>=2.20" \
                "bitsandbytes>=0.43" sentencepiece ninja packaging

# guard: pip may reinstall torchao transitively — remove again
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
print("torchao present?", importlib.util.find_spec("torchao") is not None, "(must be False)")
import transformers; print("transformers:", transformers.__version__)
print("⚠️ If the version just CHANGED: Runtime ▸ Restart session, then run from the top.")


## 3 · Config — every knob in one place

In [ ]:
import os
from dataclasses import dataclass
from typing import Optional, Tuple

@dataclass
class Config:
    drive_root: str = os.path.abspath("./opsd2_run")   # LOCAL, not Drive

    # ---- model ----
    model_id: str = "JetLM/SDAR-4B-Chat-b32"   # swap to SDAR-8B-Chat-b32 if 4B underperforms
    load_in_4bit: bool = False

    # ---- LoRA ----
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_targets: Tuple[str, ...] = ("q_proj","k_proj","v_proj","o_proj",
                                     "gate_proj","up_proj","down_proj")

    # ---- data ----
    dataset_id: str = "zwhe99/DeepMath-103K"
    n_samples: int = 2000            # raw pool to filter; yields the verified pool
    difficulty_max: Optional[int] = 5
    seed: int = 0
    col_question: str = "question"
    col_answer: str = "final_answer"
    col_difficulty: str = "difficulty"
    solution_cols: Tuple[str, ...] = ("r1_solution_1","r1_solution_2","r1_solution_3")
    n_eval_holdout: int = 8

    # ---- diffusion / OPSD (block-32 schedules) ----
    # per-step reveal = block_size // steps_per_block.
    #   teacher 16 steps -> 2 tokens/step (fine); student 4 steps -> 8 tokens/step (coarse).
    #   subset property holds: student unmask-counts {8,16,24,32} ⊆ teacher {2,4,...,32}.
    #   For the FINEST teacher (the purest experiment) set teacher_steps_per_block=32
    #   (1 token/step) — better targets, but ~2x more forwards (slower without KV-cache).
    block_size: int = 32
    student_steps_per_block: int = 4      # 8 tokens/step
    teacher_steps_per_block: int = 4      # d-OPSD: MATCHES the student (8 tok/step)
    max_gen_tokens: int = 1024            # room to reach \boxed{}; raise if boxes truncate
    temperature: float = 0.9
    top_p: float = 0.95
    rep_penalty: float = 1.1              # special tokens are EXCLUDED (see sample_token)
    noise_temp: float = 1.0               # Gumbel noise on reveal order (MaskGIT); 0 = deterministic

    # ---- d-OPSD: the teacher's ONLY advantage ----
    future_reveal_k: int = 4      # positions from LATER in the student's reveal order that are
                                  # pre-filled (with the student's own committed tokens) before
                                  # the teacher scores the current sub-step. 0 = teacher is
                                  # identical to the student -> KL collapses to ~0 (useful as a
                                  # null-control run; expect no learning signal).
    teacher_batch: int = 4        # teacher sub-step states per batched forward
    teacher_order: str = "student"  # d-OPSD requires "student" (states must be known in advance)
    time_segments: bool = True

    # ---- gates ----
    kl_clip: Optional[float] = 10.0
    repeat4_max: float = 0.60

    # ---- optimisation ----
    lr: float = 1e-5
    grad_accum: int = 4
    max_attempts_per_step: int = 12
    warmup_steps: int = 5
    opsd_n: int = 800                # OPSD trains on this many verified problems
    target_passes: int = 3           # OPSD epochs over opsd_pool (you asked 2-3)
    max_hours: Optional[float] = None

    # ---- checkpointing ----
    ckpt_every: int = 5
    keep_last_k: int = 2
    ema_beta: float = 0.9

    # ---- correction loop ----
    max_correction_attempts: int = 2

RESUME = True
cfg = Config()

import os
os.makedirs(os.path.join(cfg.drive_root, "checkpoints"), exist_ok=True)
BUFFER_PATH = os.path.join(cfg.drive_root, "correction_buffer.json")
print(cfg)


## 4 · `flash_attn` → pure-PyTorch replacements (no install/compile)
Same mechanism that got TraDo loading: a meta-path finder serves numerically-equivalent
pure-PyTorch versions of the flash-attn kernels `modeling_sdar.py` imports (fused RMSNorm +
attention via SDPA), plus the `get_imports` patch and `LossKwargs` rename bridge.

In [ ]:
import os, sys, types, importlib, importlib.abc, importlib.machinery, importlib.util
import torch
import torch.nn.functional as F

USE_FLASH_ATTN = False
for _n in [n for n in list(sys.modules) if n=="flash_attn" or n.startswith("flash_attn.")]:
    del sys.modules[_n]
for _n in [n for n in list(sys.modules) if "modeling_sdar" in n]:
    del sys.modules[_n]

import transformers.dynamic_module_utils as _dmu
_real = getattr(_dmu, "_orig_get_imports", _dmu.get_imports)
_dmu._orig_get_imports = _real
def _pgi(fn):
    imp = list(_real(fn))
    if "flash_attn" in imp and os.path.basename(str(fn)).startswith("modeling_"):
        imp = [i for i in imp if i != "flash_attn"]
    return imp
_dmu.get_imports = _pgi

import transformers.utils as _tu
for _old,_c in {"LossKwargs":["TransformersKwargs"]}.items():
    if not hasattr(_tu,_old):
        v=None
        for cand in _c:
            for mp in ("transformers.utils","transformers.processing_utils","transformers.modeling_utils","transformers"):
                try:
                    m=importlib.import_module(mp)
                    if hasattr(m,cand): v=getattr(m,cand); break
                except Exception: pass
            if v is not None: break
        setattr(_tu,_old, v if v is not None else type(_old,(dict,),{}))

def _rms_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                 eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                 zero_centered_weight=False, return_dropout_mask=False, out_dtype=None, out=None, residual_out=None):
    xdt = x.dtype
    if x1 is not None: x = x + x1
    if residual is not None:
        base = (x.float()+residual.float()) if residual_in_fp32 else (x+residual)
    else:
        base = x.float() if residual_in_fp32 else x
    nr = base
    xf = base.float()
    xn = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)
    w = (1.0+weight) if zero_centered_weight else weight
    y = xn.to(xdt) * w
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _layer_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                   eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                   zero_centered_weight=False, is_rms_norm=False, return_dropout_mask=False,
                   out_dtype=None, out=None, residual_out=None):
    if is_rms_norm:
        return _rms_norm_fn(x, weight, bias, residual, x1, weight1, bias1, eps, dropout_p,
                            rowscale, prenorm, residual_in_fp32, zero_centered_weight,
                            return_dropout_mask, out_dtype, out, residual_out)
    xdt = x.dtype
    if x1 is not None: x = x + x1
    if residual is not None:
        base = (x.float()+residual.float()) if residual_in_fp32 else (x+residual)
    else:
        base = x.float() if residual_in_fp32 else x
    nr = base; xf = base.float(); mean = xf.mean(-1, keepdim=True)
    xn = (xf-mean)*torch.rsqrt((xf-mean).pow(2).mean(-1, keepdim=True)+eps)
    w = (1.0+weight) if zero_centered_weight else weight
    y = xn.to(xdt)*w
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _expand_kv(k,v,nq):
    nk=k.shape[-2]
    if nk!=nq:
        r=nq//nk; k=k.repeat_interleave(r,dim=-2); v=v.repeat_interleave(r,dim=-2)
    return k,v

def _flash_attn_func(q,k,v,dropout_p=0.0,softmax_scale=None,causal=False,window_size=(-1,-1),
                     softcap=0.0,alibi_slopes=None,deterministic=False,return_attn_probs=False,**kw):
    k,v=_expand_kv(k,v,q.shape[-2])
    o=F.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),
                                     is_causal=causal,scale=softmax_scale,dropout_p=0.0)
    return o.transpose(1,2)

def _flash_attn_qkvpacked_func(qkv,**kw):
    q,k,v=qkv.unbind(dim=2); return _flash_attn_func(q,k,v,**kw)

def _flash_attn_varlen_func(q,k,v,cu_seqlens_q,cu_seqlens_k,max_seqlen_q=None,max_seqlen_k=None,
                            dropout_p=0.0,softmax_scale=None,causal=False,**kw):
    cq,ck=cu_seqlens_q.tolist(),cu_seqlens_k.tolist(); outs=[]
    for i in range(len(cq)-1):
        qi,ki,vi=q[cq[i]:cq[i+1]],k[ck[i]:ck[i+1]],v[ck[i]:ck[i+1]]
        ki,vi=_expand_kv(ki,vi,qi.shape[-2])
        oi=F.scaled_dot_product_attention(qi.transpose(0,1).unsqueeze(0),ki.transpose(0,1).unsqueeze(0),
                                          vi.transpose(0,1).unsqueeze(0),is_causal=causal,
                                          scale=softmax_scale,dropout_p=0.0)
        outs.append(oi.squeeze(0).transpose(0,1))
    return torch.cat(outs,0)

def _pad_input(hs,idx,b,s):
    out=hs.new_zeros(b*s,hs.shape[-1]); out[idx]=hs; return out.view(b,s,-1)
def _unpad_input(hs,am,*a,**k):
    sl=am.sum(-1).to(torch.int32); idx=torch.nonzero(am.flatten(),as_tuple=False).flatten()
    h=hs.reshape(-1,hs.shape[-1])[idx]; cu=torch.zeros(sl.numel()+1,dtype=torch.int32,device=hs.device)
    cu[1:]=torch.cumsum(sl,0); return h,idx,cu,int(sl.max().item())
def _index_first_axis(x,idx): return x.reshape(-1,*x.shape[1:])[idx]

class _RMSNormModule(torch.nn.Module):
    def __init__(self,hidden_size,eps=1e-6,**kw):
        super().__init__(); self.weight=torch.nn.Parameter(torch.ones(hidden_size)); self.eps=eps
    def forward(self,x,residual=None,prenorm=False,**kw):
        return _rms_norm_fn(x,self.weight,None,residual=residual,eps=self.eps,prenorm=prenorm)

_REG={"rms_norm_fn":_rms_norm_fn,"layer_norm_fn":_layer_norm_fn,"RMSNorm":_RMSNormModule,
      "LayerNorm":torch.nn.LayerNorm,"flash_attn_func":_flash_attn_func,
      "flash_attn_qkvpacked_func":_flash_attn_qkvpacked_func,"flash_attn_varlen_func":_flash_attn_varlen_func,
      "pad_input":_pad_input,"unpad_input":_unpad_input,"index_first_axis":_index_first_axis}
def _uns(n):
    def f(*a,**k): raise RuntimeError(f"flash_attn.{n} has no shim but was CALLED — report it.")
    return f
class _FM(types.ModuleType):
    def __getattr__(self,n):
        if n in _REG: return _REG[n]
        if n.startswith("__"): raise AttributeError(n)
        return _uns(n)
class _FF(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self,fn,path=None,target=None):
        if fn=="flash_attn" or fn.startswith("flash_attn."):
            return importlib.machinery.ModuleSpec(fn,self,is_package=True)
    def create_module(self,spec):
        m=_FM(spec.name); m.__spec__=spec; m.__path__=[]; m.__version__="0.0-shim"; return m
    def exec_module(self,m): pass

try:
    import flash_attn; USE_FLASH_ATTN=True; print("Real flash_attn present — using it.")
except ImportError:
    if not any(isinstance(f,_FF) for f in sys.meta_path): sys.meta_path.insert(0,_FF())
    import flash_attn; print("✅ pure-PyTorch flash_attn replacements active.")
_t=torch.randn(2,4,8); _w=torch.randn(8)
assert torch.allclose(_rms_norm_fn(_t,_w,eps=1e-6),
                      _t*torch.rsqrt(_t.pow(2).mean(-1,keepdim=True)+1e-6)*_w, atol=1e-5)
print("✅ RMSNorm matches reference. USE_FLASH_ATTN =", USE_FLASH_ATTN)


## 5 · torchao — already removed in §2 (kept as a no-op check)

In [ ]:
import importlib.util
print("torchao installed?", importlib.util.find_spec("torchao") is not None,
      "→ must be False (removed in §2 before transformers import).")


## 6 · Load SDAR-4B-b32, attach LoRA, define frozen teacher
Teacher = same base with adapters **disabled** (`disable_adapter()`), so it's the frozen initial
policy at zero extra memory. LoRA B-matrix starts at zero → student and teacher identical at init.

In [ ]:
import torch, transformers
from packaging import version as _v
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("transformers:", transformers.__version__)
if _v.parse(transformers.__version__.split("+")[0]) >= _v.parse("5.0.0"):
    print("⚠️ transformers 5.x — modeling_sdar will fail. Re-run §2, restart, run from top.")

tok = AutoTokenizer.from_pretrained(cfg.model_id, trust_remote_code=True)
quant = None
if cfg.load_in_4bit:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                               bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
_dt = "dtype" if _v.parse(transformers.__version__.split("+")[0]) >= _v.parse("4.56.0") else "torch_dtype"
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id, trust_remote_code=True, device_map="auto", quantization_config=quant,
    attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "sdpa", **{_dt: torch.bfloat16})
if base_model.config.pad_token_id is None:
    base_model.config.pad_token_id = tok.pad_token_id or tok.eos_token_id
print("Loaded:", type(base_model).__name__, "| attn:", getattr(base_model.config,"_attn_implementation","?"))

MASK_ID = getattr(tok, "mask_token_id", None) or 151669   # SDAR mask id
VOCAB = base_model.config.vocab_size; REAL_VOCAB = len(tok)
print("MASK_ID =", MASK_ID, "| VOCAB =", VOCAB, "| real vocab =", REAL_VOCAB, "| eos =", tok.eos_token_id)


### 6b · Discovery — how is block size / attention actually set? (verify before trusting)
Prints the model config and greps `configuration_sdar.py` / `modeling_sdar.py` for block-size and
attention handling. **What you want to see:** a `block_size`/`block_length` field in the config
equal to 32 (meaning the checkpoint's config drives block attention automatically), and an
attention path that builds a block mask. If block size is hardcoded to 4 or absent, the plain
forward may not match the trained block-32 attention — tell me what this prints.

In [ ]:
import glob, re
print("=== config fields mentioning block / attention ===")
for k,val in vars(base_model.config).items():
    if any(t in k.lower() for t in ("block","attn","window","diffus","mask")):
        print(f"  {k}: {val}")

for fname in ("configuration_sdar.py","modeling_sdar.py"):
    cand = glob.glob(os.path.expanduser(
        f"~/.cache/huggingface/modules/transformers_modules/**/{fname}"), recursive=True)
    if not cand: print(f"\n({fname} not found in cache)"); continue
    src = open(sorted(cand, key=os.path.getmtime)[-1]).read()
    print(f"\n===== {fname} — block/attention lines =====")
    for i,l in enumerate(src.splitlines()):
        if re.search(r"block_size|block_length|is_causal|attention_mask|block_diag|block_mask|def forward", l):
            print(f"  {l.strip()[:96]}")


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import os, contextlib

def _newest_ckpt():
    base = os.path.join(cfg.drive_root, "checkpoints")
    if not os.path.isdir(base): return None
    st = [int(d.split("_")[1]) for d in os.listdir(base) if d.startswith("step_") and d.split("_")[1].isdigit()]
    return os.path.join(base, f"step_{max(st)}") if st else None

if cfg.load_in_4bit:
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
else:
    base_model.gradient_checkpointing_enable(); base_model.enable_input_require_grads()

ck = _newest_ckpt() if RESUME else None
if ck and os.path.isdir(ck):
    print("Resuming adapters from:", ck)
    model = PeftModel.from_pretrained(base_model, ck, is_trainable=True)
else:
    print("Fresh LoRA (r=%d alpha=%d)." % (cfg.lora_r, cfg.lora_alpha))
    model = get_peft_model(base_model, LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_targets), bias="none", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

# ---------------------------------------------------------------------------
# CRITICAL: modeling_sdar.py's outer forward has `_update_causal_mask` COMMENTED OUT
# (verified by reading the source — see the §6b-adjacent investigation). No automatic
# causal-mask construction happens anywhere in this model. Whatever `attention_mask` we
# pass flows UNMODIFIED into SDARAttention and is used directly as SDPA's attn_mask in
# the "prefilling" branch (the "decoding" branch — no mask at all — is only valid for
# genuine cached single-token decode, which we never do). We are fully responsible for
# building the correct block-causal / bidirectional-within-block pattern ourselves.
# ---------------------------------------------------------------------------
def build_block_causal_mask(seq_len, prompt_len, block_size, device):
    """True = attend. Prompt: plain causal. Response: block-causal across blocks,
    bidirectional WITHIN a block (block(i)==block(j) trivially satisfies block(j)<=block(i)
    regardless of token order). Verified against the paper's Fig.2 description with an
    exhaustive assertion suite before ever being used against a real model."""
    idx = torch.arange(seq_len, device=device)
    is_resp = idx >= prompt_len
    blk = torch.where(is_resp, (idx - prompt_len) // block_size, idx)
    q_resp, k_resp = is_resp.unsqueeze(1), is_resp.unsqueeze(0)
    q_idx, k_idx = idx.unsqueeze(1), idx.unsqueeze(0)
    q_blk, k_blk = blk.unsqueeze(1), blk.unsqueeze(0)
    prompt_causal   = (~q_resp) & (k_idx <= q_idx)
    resp_see_prompt = q_resp & (~k_resp)
    resp_cross_blk  = q_resp & k_resp & (k_blk <= q_blk)
    return prompt_causal | resp_see_prompt | resp_cross_blk   # bool [seq_len, seq_len]

def model_logits(input_ids, prompt_len, teacher: bool = False):
    """prompt_len: length of the FIXED initial prompt for this sequence (do not pass the
    current/growing context length — capture it once, before any blocks are appended)."""
    seq_len = input_ids.shape[1]
    mask = build_block_causal_mask(seq_len, prompt_len, cfg.block_size, input_ids.device)
    ctx = model.disable_adapter() if teacher else contextlib.nullcontext()
    with ctx:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            return model(input_ids=input_ids, attention_mask=mask).logits

_p = tok("2+2=", return_tensors="pt").input_ids.to(model.device)
with torch.no_grad():
    # no response structure here -- prompt_len = full length -> plain causal, correct for a bare probe
    d = (model_logits(_p, _p.shape[1], False) - model_logits(_p, _p.shape[1], True)).abs().max().item()
print(f"student/teacher forward OK | init max|Δlogit| = {d:.2e} (≈0 now, grows as LoRA trains)")
del _p


## 7 · Data: DeepMath-103K → verified pool
Keep the first R1 solution whose own boxed answer matches `final_answer` (the teacher must be
conditioned on a *correct* solution). Prompts use the model's **own chat template**
(`apply_chat_template`) rather than a hand-written one, since SDAR-Chat ships its own.

In [ ]:
def extract_boxed(text):
    if not text: return None
    i = text.rfind("\\boxed")
    if i==-1: return None
    j = text.find("{", i)
    if j==-1: return None
    d=0
    for k in range(j,len(text)):
        if text[k]=="{": d+=1
        elif text[k]=="}":
            d-=1
            if d==0: return text[j+1:k]
    return None

def _norm(s):
    if s is None: return None
    s=str(s).strip().replace(" ","")
    for a,b in (("\\left",""),("\\right",""),("\\dfrac","\\frac"),("\\tfrac","\\frac"),("$","")): s=s.replace(a,b)
    if s.startswith("\\text{") and s.endswith("}"): s=s[6:-1]
    return s

def answers_match(pred,gold):
    a,b=_norm(pred),_norm(gold)
    if a is None or b is None: return False
    if a==b: return True
    try: return abs(float(a)-float(b))<1e-6
    except Exception: return False

def repeat4(text):
    t=text.split()
    if len(t)<4: return 0.0
    g=[tuple(t[i:i+4]) for i in range(len(t)-3)]
    return 1-len(set(g))/len(g)

assert extract_boxed(r"x \boxed{12}, y \boxed{34}")=="34"
assert answers_match(r"\dfrac{1}{2}", r"\frac{1}{2}") and not answers_match("3","4")
print("✅ answer helpers pass")


In [ ]:
from datasets import load_dataset
raw = load_dataset(cfg.dataset_id, split="train")
pool = raw.shuffle(seed=cfg.seed)
if cfg.difficulty_max is not None and cfg.col_difficulty in pool.column_names:
    pool = pool.filter(lambda e: e[cfg.col_difficulty] is not None and float(e[cfg.col_difficulty])<=cfg.difficulty_max)
    print(f"difficulty<= {cfg.difficulty_max}: {len(pool)} rows")
pool = pool.select(range(min(cfg.n_samples, len(pool))))

verified, dropped = [], 0
for e in pool:
    gold=e[cfg.col_answer]; sol=None
    for c in cfg.solution_cols:
        cand=e.get(c)
        if cand and answers_match(extract_boxed(str(cand)), gold): sol=str(cand); break
    if sol is None: dropped+=1; continue
    verified.append({"question":e[cfg.col_question], "solution":sol, "gold":str(gold)})
eval_holdout = verified[-cfg.n_eval_holdout:]; verified = verified[:-cfg.n_eval_holdout]
print(f"VERIFIED {len(verified)} | dropped {dropped} | held out {len(eval_holdout)}")

# OPSD trains on the first cfg.opsd_n verified problems (SFT's 200 are a subset -> warmup and
# OPSD share early data, reinforcing the SFT gains; note this in the paper).
opsd_pool = verified[:cfg.opsd_n]
print(f"OPSD pool: {len(opsd_pool)} problems (target {cfg.opsd_n}) x {cfg.target_passes} passes "
      f"= {len(opsd_pool)*cfg.target_passes} rollouts planned")
if len(opsd_pool) < cfg.opsd_n:
    print(f"WARNING only {len(opsd_pool)} verified (< {cfg.opsd_n}); raise cfg.n_samples in section 3 and re-run.")

In [ ]:
def build_student_prompt(q):
    return tok.apply_chat_template(
        [{"role":"user","content": f"{q}\nPlease reason step by step, and put your final answer within \\boxed{{}}."}],
        tokenize=False, add_generation_prompt=True)

def build_teacher_prompt(q, sol, failure_note=""):
    content = (f"{q}\n\nHere is a reference solution:\n{sol}\n\n"
               f"After understanding the reference solution, solve the problem yourself.{failure_note}\n"
               f"Please reason step by step, and put your final answer within \\boxed{{}}.")
    return tok.apply_chat_template([{"role":"user","content":content}],
                                   tokenize=False, add_generation_prompt=True)

def correction_note(wrong, gold):
    return (f"\nNote: a previous attempt concluded \\boxed{{{wrong}}}, which is INCORRECT. "
            f"Identify the error and avoid it; the correct final answer is {gold}.")
print(build_student_prompt(verified[0]["question"])[:200])


## 7c · Block-size ceiling check — run this BEFORE §7.5/§8

**Why before committing to SFT+OPSD.** The SDAR paper reports that small models (1.7B, 4B) are
**sensitive to block-size increases**, with degradation "pronounced for B > 4" — but also that
**B∈{8,16} often matches or beats B=4 on math tasks specifically** (GSM8K, MathBench), while
**B=32 is where task-dependent trade-offs concentrate**. That's a direct, math-relevant warning
about the exact checkpoint this notebook defaults to. Rather than discover this after hours of
SFT+OPSD, we measure it now, directly, on your own held-out DeepMath problems.

**Method.** For each candidate block size we run the model's own **performance ceiling**: greedy,
**exhaustive** denoising (1 token revealed per step — the paper's own "static decoding...
establishing the performance ceiling for the given architecture"). This isolates *"is the
checkpoint itself capable at this block size"* from *"is our coarse OPSD schedule too aggressive"*
— two different questions our later results would otherwise conflate.

**A useful cost property:** full-ceiling denoising costs exactly `budget` forward passes
*regardless of block size* (block_size steps × budget/block_size blocks = budget, always) — so
this comparison is naturally fair in compute, not just in intent.

**What to do with the result:** if block-16 holds up close to block-4 and block-32 shows a real
drop (as the paper predicts for math), switch `cfg.model_id` to `JetLM/SDAR-4B-Chat-b16` below
and re-run from §6 with an adjusted schedule (e.g. teacher 8 steps/block, student 2 steps/block)
before investing in SFT+OPSD. If block-16 *also* degrades meaningfully, that's concrete,
your-own-data evidence for escalating to `SDAR-8B-Chat-b32` with your professor.


In [ ]:
import torch, gc

@torch.no_grad()
def ceiling_eval(pool, n, block_size, mask_id, invalid_mask, budget=768, label="",
                 forward_fn=None, model_=None, tok_=None):
    """Greedy, EXHAUSTIVE denoising (1 token/step = block_size steps/block) — the model's own
    performance ceiling. Isolates checkpoint capability from our OPSD schedule.
    Pass EITHER model_ (a real nn.Module; we build+pass the correct block-causal mask
    ourselves, same as model_logits does) OR forward_fn(ids, prompt_len)->logits for the
    primary model (routed through model_logits so its own masking applies)."""
    tok_ = tok_ or tok
    was_training = False
    if model_ is not None:
        was_training = model_.training
        model_.eval()
        def fwd(ids, plen):
            mask = build_block_causal_mask(ids.shape[1], plen, block_size, ids.device)
            return model_(input_ids=ids, attention_mask=mask).logits
    else:
        fwd = forward_fn

    n_eval = min(n, len(pool))
    correct, rows = 0, []
    for i in range(n_eval):
        ex = pool[i]
        ids = tok_(build_student_prompt(ex["question"]), return_tensors="pt").input_ids.to(
            model_.device if model_ is not None else model.device)
        start, gen = ids.shape[1], 0
        prompt_len = start   # FIXED for this rollout; do not update inside the loop below
        while gen < budget:
            work = torch.cat([ids, torch.full((1, block_size), mask_id,
                                               dtype=ids.dtype, device=ids.device)], dim=1)
            pos = list(range(ids.shape[1], ids.shape[1] + block_size))
            remaining = set(range(block_size))
            while remaining:                                   # 1 token/step: the ceiling
                logits = fwd(work, prompt_len)[0, pos, :].float().masked_fill(invalid_mask, float("-inf"))
                conf = torch.log_softmax(logits, dim=-1).max(-1).values
                j = max(remaining, key=lambda j: conf[j].item())
                work[0, pos[j]] = int(logits[j].argmax())
                remaining.discard(j)
            ids = torch.cat([ids, work[:, pos]], dim=1)
            gen += block_size
            if tok_.eos_token_id is not None and (work[:, pos] == tok_.eos_token_id).any():
                break
        txt = tok_.decode(ids[0, start:].tolist(), skip_special_tokens=True)
        pred = extract_boxed(txt)
        ok = answers_match(pred, ex["gold"]) if pred else False
        correct += ok
        rows.append((i, pred, ex["gold"], ok))
        print(f"  [{label}] [{i+1}/{n_eval}] pred={str(pred):>8} gold={ex['gold']:>8} "
              f"{'✓' if ok else '·'}")
    if model_ is not None and was_training: model_.train()
    acc = correct / max(1, n_eval)
    print(f"  [{label}] ceiling accuracy: {correct}/{n_eval} = {acc:.2f}\n")
    return acc, rows

def _vocab_guard_for(model_, tok_):
    """Self-contained vocab guard (same math as §7.5's, computed per-model so a different
    checkpoint's vocab_size can't silently reuse the wrong mask)."""
    vs = model_.config.vocab_size
    rv = len(tok_)
    inv = torch.zeros(vs, dtype=torch.bool)
    if rv < vs: inv[rv:] = True
    mid = getattr(tok_, "mask_token_id", None) or 151669
    inv[mid] = True
    return inv.to(model_.device), mid

print("ceiling_eval + vocab guard helper ready (block-causal mask applied for every model).")


In [ ]:
import gc, torch, transformers
from packaging import version as _v
from transformers import AutoModelForCausalLM

# n / budget kept small deliberately: this is a decisive smoke check, not a full eval.
CEIL_N = 6
CEIL_BUDGET = 768
# comparison block sizes. cfg.model_id's own block size is checked via the ALREADY-LOADED
# `base_model` (through model_logits(teacher=True) to bypass LoRA cleanly -- LoRA is
# zero-init at this point anyway, so this is equivalent to the pristine checkpoint).
COMPARE = {16: "JetLM/SDAR-4B-Chat-b16", 4: "JetLM/SDAR-4B-Chat"}

results = {}

# --- current primary model (block-cfg.block_size), via the frozen-teacher forward path ---
inv32, mid32 = _vocab_guard_for(base_model, tok)
acc32, _ = ceiling_eval(eval_holdout, CEIL_N, cfg.block_size, mid32, inv32, budget=CEIL_BUDGET,
                        label=f"b{cfg.block_size} (current)",
                        forward_fn=lambda ids, plen: model_logits(ids, plen, teacher=True))
results[cfg.block_size] = acc32

# --- comparison models: load fresh, eval, free before loading the next one ---
_dt = "dtype" if _v.parse(transformers.__version__.split("+")[0]) >= _v.parse("4.56.0") else "torch_dtype"
for bsize, mid_str in COMPARE.items():
    print(f"Loading {mid_str} for comparison...")
    cmp_model = AutoModelForCausalLM.from_pretrained(
        mid_str, trust_remote_code=True, device_map="auto",
        attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "sdpa", **{_dt: torch.bfloat16})
    inv, mid = _vocab_guard_for(cmp_model, tok)
    acc, _ = ceiling_eval(eval_holdout, CEIL_N, bsize, mid, inv, budget=CEIL_BUDGET,
                          label=f"b{bsize}", model_=cmp_model, tok_=tok)
    results[bsize] = acc
    del cmp_model; gc.collect(); torch.cuda.empty_cache()   # free before next load

print("="*50)
hdr = "block size"; hdr2 = "ceiling acc"
print(f"{hdr:>10} | {hdr2:>11} | vs current(b{cfg.block_size})")
for bs in sorted(results, reverse=True):
    tag = "  <- current default" if bs == cfg.block_size else \
          f"  ({results[bs]-results[cfg.block_size]:+.2f})"
    print(f"{bs:>10} | {results[bs]:>11.2f} |{tag}")


**Reading the table:** `CEIL_N=6` is a smoke check, not a statistically solid benchmark —
treat a 1-2 problem swing as noise, and a consistent 3+ problem gap as signal. If block-16 is
within noise of block-4 and block-32 is meaningfully behind both, switch `cfg.model_id` to
`JetLM/SDAR-4B-Chat-b16` in §3, re-run from §6 (schedule: e.g. `teacher_steps_per_block=8`,
`student_steps_per_block=2` for a 4:1 ratio at block-16), and treat this cell's numbers as your
evidence trail for the write-up. If you want a more statistically solid read before deciding,
raise `CEIL_N` to 15-20 and re-run — still cheap, since cost is `budget`-bound, not block-size-bound.

## 7.5 · Diffusion-aware SFT warmup (run BEFORE OPSD)
Force-teaches the student — under its own *unprivileged* prompt — to reconstruct verified-correct
solutions using block-masked cross-entropy. **`SFT_TOKENS_PER_STEP` is set to match the student's
OPSD reveal (8 tokens/step at block-32)** so SFT practises exactly the conditional OPSD will use.
No teacher, no sampling, no gates — the target is the reference solution. Backward is **per block**
(flat memory regardless of solution length).

In [ ]:
from dataclasses import dataclass
@dataclass
class SFTConfig:
    n_samples: int = 200          # SFT warmup pool (you asked 200)
    epochs: int = 6               # you asked 5-6
    tokens_per_step: int = 8       # MATCHES student OPSD reveal at block-32 (block_size//student_steps = 32//4)
    max_target_tokens: int = 1024
    grad_accum: int = 4
    lr: float = 1e-5
    warmup_steps: int = 10
    ckpt_every_steps: int = 25
    keep_last_k: int = 2
    log_every_samples: int = 10
    ema_beta: float = 0.9
    eval_n: int = 6
sftcfg = SFTConfig()
SFT_CKPT_DIR = os.path.join(cfg.drive_root, f"checkpoints_sft_{sftcfg.tokens_per_step}tok")
os.makedirs(SFT_CKPT_DIR, exist_ok=True)
sft_pool = verified[:sftcfg.n_samples]
import math
_spe = math.ceil(len(sft_pool)/sftcfg.grad_accum)
print(f"SFT: {len(sft_pool)} samples × {sftcfg.epochs} epochs | {sftcfg.tokens_per_step} tok/step "
      f"| block {cfg.block_size} | ~{_spe*sftcfg.epochs} steps | -> {SFT_CKPT_DIR}")


In [ ]:
# vocab guard (shared by SFT and OPSD): mask padded dead slots + MASK token itself
import torch
_inv = torch.zeros(VOCAB, dtype=torch.bool)
if REAL_VOCAB < VOCAB: _inv[REAL_VOCAB:] = True
_inv[MASK_ID] = True
INVALID_MASK = _inv.to(model.device)
def _append_masks(ids, b):
    return torch.cat([ids, torch.full((1,b), MASK_ID, dtype=ids.dtype, device=ids.device)], dim=1)
print(f"vocab guard: {int(INVALID_MASK.sum())} invalid ids masked")


In [ ]:
import torch, torch.nn.functional as F
def sft_sequence_loss(question, solution, tokens_per_step, loss_scale):
    """Block-masked CE against the gold solution; backward() PER BLOCK (flat memory)."""
    prompt_ids = tok(build_student_prompt(question), return_tensors="pt").input_ids.to(model.device)
    sol_ids = tok(solution, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    if tok.eos_token_id is not None:
        sol_ids = torch.cat([sol_ids, torch.tensor([[tok.eos_token_id]], device=model.device, dtype=sol_ids.dtype)], dim=1)
    sol_ids = sol_ids[:, :sftcfg.max_target_tokens]
    B, per = cfg.block_size, tokens_per_step
    ctx = prompt_ids
    prompt_len = prompt_ids.shape[1]     # FIXED for the whole rollout -- do NOT use ctx.shape[1]
                                         # below (that grows every block; the mask needs the
                                         # ORIGINAL prompt/response boundary, always)
    running, npos, nblk = 0.0, 0, 0
    for b0 in range(0, sol_ids.shape[1], B):
        gold = sol_ids[:, b0:b0+B]; bs = gold.shape[1]
        if bs==0: break
        revealed = torch.full((1,bs), MASK_ID, device=model.device, dtype=ctx.dtype)
        remaining = list(range(bs)); nblk += 1
        bloss = ctx.new_zeros((), dtype=torch.float32); bpos = 0
        while remaining:
            work = torch.cat([ctx, revealed], dim=1)
            pos = list(range(ctx.shape[1], ctx.shape[1]+bs))
            logits = model_logits(work, prompt_len, teacher=False)[0, pos, :].float().masked_fill(INVALID_MASK, float("-inf"))
            mp = [p for p in range(bs) if int(revealed[0,p])==MASK_ID]
            bloss = bloss + F.cross_entropy(logits[mp], gold[0, mp], reduction="sum"); bpos += len(mp)
            for p in remaining[:per]: revealed[0,p] = gold[0,p]
            remaining = remaining[per:]
        ((bloss / max(1,bpos)) * loss_scale).backward()
        running += bloss.item()/max(1,bpos); npos += bpos
        ctx = torch.cat([ctx, gold.detach()], dim=1)
    return running/max(1,nblk), npos, nblk
print("SFT loss ready (per-block backward, correct block-causal mask via fixed prompt_len).")


In [ ]:
@torch.no_grad()
def sft_eval(pool, n, budget=1024, tag=""):
    was = model.training; model.eval()
    ok_n, r4s, boxed = 0, [], 0
    print(f"\n=== EVAL {tag} ({n} problems, greedy) ===")
    for i in range(min(n,len(pool))):
        ex=pool[i]; ids=tok(build_student_prompt(ex["question"]),return_tensors="pt").input_ids.to(model.device)
        start,B,gen=ids.shape[1],cfg.block_size,0
        while gen<budget:
            work=_append_masks(ids,B); pos=list(range(ids.shape[1],ids.shape[1]+B))
            lg=model_logits(work,start,teacher=False)[0,pos,:].float().masked_fill(INVALID_MASK,float("-inf"))
            commit=lg.argmax(-1).view(1,-1); ids=torch.cat([ids,commit],dim=1); gen+=B
            if tok.eos_token_id is not None and (commit==tok.eos_token_id).any(): break
        g=[t for t in ids[0,start:].tolist() if t<REAL_VOCAB]; txt=tok.decode(g,skip_special_tokens=True)
        pred=extract_boxed(txt); r4=repeat4(txt); ok=answers_match(pred,ex["gold"]) if pred else False
        ok_n+=ok; r4s.append(r4); boxed+=(pred is not None)
        print(f"  [{i+1}] {len(g):5d} tok | box={str(pred):>8} {'✓' if ok else '·'} | rep4 {r4:.3f} | gold {ex['gold']}")
    if was: model.train()
    print(f"  --> correct {ok_n}/{min(n,len(pool))} | boxed {boxed}/{min(n,len(pool))} | mean rep4 {sum(r4s)/max(1,len(r4s)):.3f}")
    return ok_n
_ = sft_eval(eval_holdout, sftcfg.eval_n, tag="BEFORE SFT")


In [ ]:
import torch, os, shutil, time, math
from transformers import get_cosine_schedule_with_warmup
sft_trainable = [p for p in model.parameters() if p.requires_grad]
sft_opt = torch.optim.AdamW(sft_trainable, lr=sftcfg.lr)
SFT_TOTAL = math.ceil(len(sft_pool)/sftcfg.grad_accum)*sftcfg.epochs
sft_sched = get_cosine_schedule_with_warmup(sft_opt, sftcfg.warmup_steps, SFT_TOTAL)

def sft_save(step, epoch, ema, best=False):
    d=os.path.join(SFT_CKPT_DIR, "best" if best else f"step_{step}"); os.makedirs(d, exist_ok=True)
    model.save_pretrained(d)
    torch.save({"step":step,"epoch":epoch,"ema":ema,"opt":sft_opt.state_dict(),"sched":sft_sched.state_dict()},
               os.path.join(d,"sft_state.pt"))
    if not best:
        st=sorted(int(x.split("_")[1]) for x in os.listdir(SFT_CKPT_DIR) if x.startswith("step_") and x.split("_")[1].isdigit())
        for o in st[:-sftcfg.keep_last_k]: shutil.rmtree(os.path.join(SFT_CKPT_DIR,f"step_{o}"), ignore_errors=True)
    print(f"  {'🏆 best' if best else '💾'} SFT ckpt → {d}")

model.train(); step, ema, best_ema = 0, None, float("inf"); t0all=time.time()
print(f"SFT START | {SFT_TOTAL} total steps\n")
try:
    for epoch in range(sftcfg.epochs):
        print(f"╔══ EPOCH {epoch+1}/{sftcfg.epochs} ══╗")
        rl=[]; sft_opt.zero_grad(set_to_none=True); acc=0
        for idx, ex in enumerate(sft_pool):
            t0=time.time()
            loss, npos, nblk = sft_sequence_loss(ex["question"], ex["solution"], sftcfg.tokens_per_step,
                                                 loss_scale=1.0/sftcfg.grad_accum)
            acc+=1; rl.append(loss); dt=time.time()-t0
            bi=ex["solution"].rfind("\\boxed")
            tb=len(tok(ex["solution"][:bi+40], add_special_tokens=False).input_ids) if bi!=-1 else None
            print(f"  e{epoch+1}[{idx+1:03d}/{len(sft_pool)}] {dt:5.1f}s | {nblk:2d}blk {npos:4d}pos | "
                  f"{'gold-box@'+str(tb) if tb else 'no-box':>12} | loss {loss:.4f}")
            if acc==sftcfg.grad_accum:
                torch.nn.utils.clip_grad_norm_(sft_trainable,1.0); sft_opt.step(); sft_sched.step()
                sft_opt.zero_grad(set_to_none=True); acc=0; step+=1
                a=sum(rl[-sftcfg.grad_accum:])/sftcfg.grad_accum
                ema=a if ema is None else sftcfg.ema_beta*ema+(1-sftcfg.ema_beta)*a
                best=ema<best_ema
                if best: best_ema=ema
                if step%sftcfg.ckpt_every_steps==0: sft_save(step,epoch,ema)
                if best and step>=sftcfg.warmup_steps: sft_save(step,epoch,ema,best=True)
            if (idx+1)%sftcfg.log_every_samples==0:
                r=rl[-sftcfg.log_every_samples:]
                print(f"    · step {step:3d} | mean(last {len(r)}) {sum(r)/len(r):.4f} | "
                      f"ema {ema if ema else float('nan'):.4f} | lr {sft_sched.get_last_lr()[0]:.2e} | "
                      f"{(time.time()-t0all)/60:.1f} min")
        if acc>0:
            torch.nn.utils.clip_grad_norm_(sft_trainable,1.0); sft_opt.step(); sft_sched.step()
            sft_opt.zero_grad(set_to_none=True); step+=1
        print(f"╚══ epoch {epoch+1} done | ema {ema:.4f} | {(time.time()-t0all)/60:.1f} min ══╝\n")
        sft_save(step, epoch, ema)
except KeyboardInterrupt:
    print("\n⏹ SFT interrupted — saving.")
finally:
    sft_save(step, sftcfg.epochs-1, ema if ema is not None else float("nan"))
    print(f"\nSFT done/paused at step {step}, best ema {best_ema:.4f} → {SFT_CKPT_DIR}/best")


In [ ]:
_ = sft_eval(eval_holdout, sftcfg.eval_n, tag="AFTER SFT")
print("\nRead the before/after delta: more boxes reached + lower rep4 = SFT helped. If the model "
      "still can't reach boxes at block-32, that's the SDAR-4B-at-large-block weakness (consider 8B-b32).")


## 8 · d-OPSD core — teacher = student + lookahead

Student and teacher both reveal `block_size//student_steps_per_block` = **8 tokens/step** over 4
sub-steps, from the **same prompt**. At teacher sub-step `s` the teacher additionally sees
`cfg.future_reveal_k` positions drawn from *later* in the student's reveal order, filled with the
tokens the student committed — the lookahead that makes its conditional better-informed.

Positions scored at sub-step `s` are exactly the ones the student revealed at `s`, so the KL is
always computed on matched positions. The future positions are taken from
`reveal_order[(s+1)*per : (s+1)*per + k]`, i.e. strictly beyond what the student had at that
point, and never overlap the positions being scored.

Set `future_reveal_k=0` for a null control: the teacher becomes the student exactly, KL ≈ 0, and
no learning should occur. Worth running once for 2-3 problems — if loss is not ≈0 there,
something is wrong with the matching.

In [ ]:
import torch, time
def _mask_invalid(l): return l.masked_fill(INVALID_MASK, float("-inf"))

def select_positions(logp, remaining, k, noise_temp):
    conf = logp.max(dim=-1).values
    if noise_temp>0:
        u = torch.rand_like(conf).clamp_min(1e-9)
        conf = conf + noise_temp*(-torch.log(-torch.log(u)+1e-9))
    return sorted(remaining, key=lambda j: conf[j].item(), reverse=True)[:k]

# special tokens EXCLUDED from repetition penalty (prevents the early-EOS pathology)
_SPECIAL = set(tok.all_special_ids or [])
_SPECIAL.add(MASK_ID)
if tok.eos_token_id is not None: _SPECIAL.add(tok.eos_token_id)
_SPECIAL_T = torch.tensor(sorted(_SPECIAL), device=model.device)

def sample_token(logits_j, prev_ids):
    lg = logits_j.detach().clone()
    if cfg.rep_penalty != 1.0 and prev_ids is not None and prev_ids.numel()>0:
        uniq = torch.unique(prev_ids)
        mask = ~torch.isin(uniq, _SPECIAL_T)     # do NOT penalize special tokens
        uniq = uniq[mask]
        if uniq.numel()>0:
            s = lg[uniq]; lg[uniq] = torch.where(s<0, s*cfg.rep_penalty, s/cfg.rep_penalty)
    if cfg.temperature<=0: return int(lg.argmax())
    probs = (lg/cfg.temperature).softmax(-1)
    if cfg.top_p<1.0:
        sp,si = probs.sort(descending=True)
        keep = (sp.cumsum(-1)-sp) < cfg.top_p
        sp = torch.where(keep, sp, torch.zeros_like(sp)); sp = sp/sp.sum()
        return int(si[torch.multinomial(sp,1)])
    return int(torch.multinomial(probs,1))

def run_block(student_ids, teacher_ids, prev_gen_ids, s_prompt_len, t_prompt_len):
    # d-OPSD: teacher_ids IS student_ids and t_prompt_len IS s_prompt_len (same prompt).
    # The arguments are kept for signature compatibility with the rest of the notebook.
    B = cfg.block_size
    _t = {"student": 0.0, "teacher": 0.0}
    # STUDENT: multi-sub-step, grad ON, stochastic
    _t0 = time.time()
    s_work=_append_masks(student_ids,B); s_base=student_ids.shape[1]; s_pos=list(range(s_base,s_base+B))
    s_logp=[None]*B; committed=[None]*B; remaining=set(range(B))
    reveal_order=[]
    per_s=max(1, B//cfg.student_steps_per_block)
    while remaining:
        logits=model_logits(s_work, s_prompt_len, teacher=False)[0,s_pos,:]; logits=_mask_invalid(logits.float())
        logp=torch.log_softmax(logits,dim=-1)
        for j in select_positions(logp, remaining, per_s, cfg.noise_temp):
            s_logp[j]=logp[j]; committed[j]=sample_token(logits[j], prev_gen_ids)
            s_work[0,s_pos[j]]=committed[j]; remaining.discard(j)
            reveal_order.append(j)
    student_logp=torch.stack(s_logp,0)
    committed_t=torch.tensor(committed, device=student_ids.device, dtype=student_ids.dtype)
    _t["student"] = time.time()-_t0

    # TEACHER: same prompt, same rate, grad OFF, adapters OFF, + FUTURE LOOKAHEAD
    _t0 = time.time()
    per_t = max(1, B//cfg.teacher_steps_per_block)
    k = int(cfg.future_reveal_k)
    def _future(s):
        # positions strictly AFTER the ones the student had at sub-step s, and after the
        # positions being scored at s -> never leaks a token the teacher is scoring.
        start = (s + 1) * per_t
        return reveal_order[start : start + k] if k > 0 else []
    teacher_logp = teacher_logp_batched(teacher_ids, t_prompt_len, committed_t,
                                        reveal_order, per_t, extra_reveal=_future)
    _t["teacher"] = time.time()-_t0
    return student_logp, teacher_logp, committed_t, _t

def distill_loss(s_logp, t_logp):
    p=t_logp.exp(); kl=torch.where(p>0, p*(t_logp-s_logp), torch.zeros_like(p)).sum(-1)
    if cfg.kl_clip is not None: kl=kl.clamp(max=cfg.kl_clip)
    return kl.mean()
assert cfg.teacher_order == "student", "d-OPSD needs teacher_order='student'"
print(f"d-OPSD core ready | student {cfg.block_size//cfg.student_steps_per_block} tok/step "
      f"| teacher {cfg.block_size//cfg.teacher_steps_per_block} tok/step "
      f"| future_reveal_k={cfg.future_reveal_k}")


In [ ]:
# ---------------------------------------------------------------------------
# Batched teacher scoring — the repair for the phase-2 wall-clock.
#
# THE COST, MEASURED IN FORWARDS (block_size=32, max_gen_tokens=1536 -> 48 blocks):
#   student : student_steps_per_block(4) x 48 =  192 forwards, grad ON  (+48 backwards)
#   teacher : teacher_steps_per_block(16) x 48 = 768 forwards, grad OFF
#   -> the teacher is ~70% of all forward work, and target_passes=3 multiplies everything.
# Screening/inference at 8 tok/step is only 192 forwards per problem, which is why
# "training = inference twice" undercounted by roughly 5-6x per pass.
#
# WHY IT CAN BE BATCHED: the teacher re-scores tokens the STUDENT already committed.
# Once the student finishes a block, `committed_t` is fully known, so every teacher
# sub-step input state can be constructed up front instead of discovered sequentially.
# n_states states then run as ceil(n_states/teacher_batch) batched forwards.
#
# THE ONE SEMANTIC CHANGE (disclose this in the paper): batching requires the teacher's
# reveal ORDER to be fixed in advance, so it uses the student's own reveal order rather
# than re-ranking by the teacher's own confidence at each sub-step. The set of tokens
# conditioned on at each state is unchanged; only the order in which they are added
# differs. Set cfg.teacher_order="teacher_conf" to fall back to the original sequential
# path (exact old semantics, ~teacher_batch times slower) if you want an A/B.
# ---------------------------------------------------------------------------
import math, torch

def model_logits_batched(input_ids, prompt_len, teacher: bool = False):
    """Same as model_logits but input_ids may be [n, L]; the [L, L] bool mask broadcasts."""
    seq_len = input_ids.shape[1]
    mask = build_block_causal_mask(seq_len, prompt_len, cfg.block_size, input_ids.device)
    ctx = model.disable_adapter() if teacher else contextlib.nullcontext()
    with ctx:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            return model(input_ids=input_ids, attention_mask=mask).logits

@torch.no_grad()
def teacher_logp_batched(teacher_ids, t_prompt_len, committed_t, reveal_order, per_t,
                         extra_reveal=None):
    """All teacher sub-step states at once.

    teacher_ids  : context BEFORE this block (prompt + previously committed blocks)
    committed_t  : [B] tokens the student committed for this block
    reveal_order : list of block-local positions, in the order they get revealed
    per_t        : tokens the teacher reveals per sub-step
    extra_reveal : optional callable(state_index) -> list of extra block-local positions
                   to ALSO fill with committed tokens at that state. This is the d-OPSD
                   future-token hook; None for the privileged-prompt variant.
    Returns [B, V] log-probs, one row per block position, taken at the state just
    before that position was revealed (same conditioning as the sequential version).
    """
    B = cfg.block_size
    t_base = teacher_ids.shape[1]
    base_state = _append_masks(teacher_ids, B)[0]
    n_states = math.ceil(B / per_t)

    states, scored = [], []
    for s in range(n_states):
        w = base_state.clone()
        for j in reveal_order[: s * per_t]:                 # everything revealed so far
            w[t_base + j] = committed_t[j]
        if extra_reveal is not None:                        # d-OPSD: peek at the future
            for j in extra_reveal(s):
                w[t_base + j] = committed_t[j]
        states.append(w)
        scored.append(reveal_order[s * per_t : (s + 1) * per_t])

    out = [None] * B
    bs = max(1, int(cfg.teacher_batch))
    for b0 in range(0, n_states, bs):
        batch = torch.stack(states[b0 : b0 + bs])
        lg_all = model_logits_batched(batch, t_prompt_len, teacher=True)
        for r in range(batch.shape[0]):
            positions = scored[b0 + r]
            if not positions: continue
            rows = [t_base + j for j in positions]
            lp = torch.log_softmax(_mask_invalid(lg_all[r, rows, :].float()), dim=-1)
            for m, j in enumerate(positions):
                out[j] = lp[m]
        del lg_all
    assert all(o is not None for o in out), "some block position never got a teacher target"
    return torch.stack(out, 0)
print("batched teacher ready | teacher_batch =", cfg.teacher_batch,
      "| teacher_order =", cfg.teacher_order)


In [ ]:
import time
def _tokens_to_box(text):
    i=text.rfind("\\boxed")
    if i==-1: return None
    j=text.find("{",i)
    if j==-1: return None
    d=0
    for k in range(j,len(text)):
        if text[k]=="{": d+=1
        elif text[k]=="}":
            d-=1
            if d==0: return len(tok(text[:k+1], add_special_tokens=False).input_ids)
    return None

def rollout_and_backward(question, solution, gold, failure_note=""):
    s_ids=tok(build_student_prompt(question), return_tensors="pt").input_ids.to(model.device)
    # d-OPSD: teacher uses the SAME prompt as the student (no reference solution).
    # `solution` / `failure_note` are accepted but unused, so the pool format and the
    # correction loop in §14 keep working unchanged.
    t_ids=s_ids.clone()
    # gs/gt: FIXED prompt lengths for student/teacher (teacher's is longer -- it embeds the
    # privileged solution). Captured once, before any blocks are appended, then held constant
    # for the whole rollout -- this is what makes the block-causal mask correct at every step.
    gs=s_ids.shape[1]; gt=t_ids.shape[1]; gen, running, nblk = 0, 0.0, 0
    T={'student':0.0,'teacher':0.0,'backward':0.0}
    while gen<cfg.max_gen_tokens:
        s_logp, t_logp, committed, _t = run_block(s_ids, t_ids, s_ids[0, gs:], gs, gt)
        T['student']+=_t['student']; T['teacher']+=_t['teacher']
        loss = distill_loss(s_logp, t_logp)/cfg.grad_accum
        _b=time.time(); loss.backward(); T['backward']+=time.time()-_b
        running += loss.item()*cfg.grad_accum; nblk+=1
        c=committed.view(1,-1); s_ids=torch.cat([s_ids,c],dim=1); t_ids=torch.cat([t_ids,c],dim=1)
        gen+=cfg.block_size
        if tok.eos_token_id is not None and (c==tok.eos_token_id).any(): break
    safe=[t for t in s_ids[0,gs:].tolist() if t<REAL_VOCAB]; txt=tok.decode(safe,skip_special_tokens=True)
    pred=extract_boxed(txt); r4=repeat4(txt)
    if r4>cfg.repeat4_max: outcome="repetitive"
    elif pred is None: outcome="unverifiable"
    elif answers_match(pred,gold): outcome="correct"
    else: outcome="wrong"
    return {"loss":running/max(1,nblk),"n_blocks":nblk,"gen_tokens":len(safe),"text":txt,
            "pred":pred,"repeat4":r4,"tokens_to_box":_tokens_to_box(txt),"outcome":outcome,
            "t_student":T["student"],"t_teacher":T["teacher"],"t_backward":T["backward"]}
print("rollout_and_backward ready.")


## 9 · Answer-gated rollback — snapshot **gradients** (not weights)

In [ ]:
def snapshot_grads():
    return {n:(p.grad.detach().clone() if p.grad is not None else None)
            for n,p in model.named_parameters() if p.requires_grad}
def restore_grads(snap):
    for n,p in model.named_parameters():
        if not p.requires_grad: continue
        g=snap.get(n)
        if g is None: p.grad=None
        else:
            if p.grad is None: p.grad=g.clone()
            else: p.grad.copy_(g)
print("grad snapshot/restore ready.")


## 10 · Smoke test — mechanics only (tiny budget)

In [ ]:
import traceback
model.train(); _sv=cfg.max_gen_tokens; cfg.max_gen_tokens=cfg.block_size  # 1 block
try:
    model.zero_grad(set_to_none=True)
    snap0=snapshot_grads()
    r=rollout_and_backward(verified[0]["question"], verified[0]["solution"], verified[0]["gold"])
    g=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
    assert len(g)>0, "no LoRA grads"; assert torch.isfinite(torch.tensor(r["loss"]))
    print(f"rollout: loss={r['loss']:.4f} blocks={r['n_blocks']} outcome={r['outcome']}")
    restore_grads(snap0)
    assert all(p.grad is None for p in model.parameters() if p.requires_grad), "restore failed"
    print("✅ smoke OK — grads flow, rollback exact.")
except Exception:
    print("❌ smoke failed:\n"); traceback.print_exc()
finally:
    cfg.max_gen_tokens=_sv; model.zero_grad(set_to_none=True)


## 11 · Throughput probe — measure s/sample before committing

In [ ]:
import time
model.train(); model.zero_grad(set_to_none=True); times=[]
print("Per-problem cost breakdown.\n")
for i in range(2):
    snap=snapshot_grads(); t0=time.time()
    r=rollout_and_backward(**opsd_pool[10+i]); dt=time.time()-t0; restore_grads(snap)
    times.append(dt); tot=max(dt,1e-9)
    print(f"probe {i+1}: {dt:6.1f}s | {r['n_blocks']} blk | {r['gen_tokens']} tok | "
          f"{r['outcome']} | loss {r['loss']:.4f}")
    print(f"   student  {r['t_student']:6.1f}s ({r['t_student']/tot:5.1%})   "
          f"teacher {r['t_teacher']:6.1f}s ({r['t_teacher']/tot:5.1%})   "
          f"backward {r['t_backward']:6.1f}s ({r['t_backward']/tot:5.1%})")
model.zero_grad(set_to_none=True)
avg=sum(times)/len(times)
print(f"\navg {avg:.1f}s/sample -> {len(opsd_pool)} problems x {cfg.target_passes} passes "
      f"= {avg*len(opsd_pool)*cfg.target_passes/3600:.1f} h")
print("\nSANITY: set cfg.future_reveal_k=0 and re-run this cell — loss should collapse to ~0")
print("(teacher becomes the student exactly). If it does not, the KL positions are misaligned.")


## 12 · Optimizer, checkpoint, buffer persistence, resume

In [ ]:
import torch, os, shutil, json, math
from transformers import get_cosine_schedule_with_warmup
trainable=[p for p in model.parameters() if p.requires_grad]
opt=torch.optim.AdamW(trainable, lr=cfg.lr)
TOTAL=max(1, math.ceil(cfg.target_passes*len(opsd_pool)/cfg.grad_accum))
sched=get_cosine_schedule_with_warmup(opt, cfg.warmup_steps, TOTAL)
print("planned OPSD steps:", TOTAL)

def save_ckpt(step, st):
    d=os.path.join(cfg.drive_root,"checkpoints",f"step_{step}"); os.makedirs(d, exist_ok=True)
    model.save_pretrained(d)
    torch.save({"opt":opt.state_dict(),"sched":sched.state_dict(),**st}, os.path.join(d,"trainer_state.pt"))
    base=os.path.join(cfg.drive_root,"checkpoints")
    steps=sorted(int(x.split("_")[1]) for x in os.listdir(base) if x.startswith("step_") and x.split("_")[1].isdigit())
    for o in steps[:-cfg.keep_last_k]: shutil.rmtree(os.path.join(base,f"step_{o}"), ignore_errors=True)
    print(f"  💾 → {d}")
def save_best(step, ema):
    d=os.path.join(cfg.drive_root,"checkpoints","best"); os.makedirs(d, exist_ok=True)
    model.save_pretrained(d); torch.save({"step":step,"smoothed_loss":ema}, os.path.join(d,"best_meta.pt"))
    print(f"  🏆 best (ema {ema:.4f}) → {d}")
def load_buffer():
    return json.load(open(BUFFER_PATH)) if os.path.isfile(BUFFER_PATH) else []
def save_buffer(b): json.dump(b, open(BUFFER_PATH,"w"))

state={"step":0,"pass_idx":0,"cursor":0,"ema":None,"best_ema":float("inf"),
       "n_correct":0,"n_wrong":0,"n_unverifiable":0,"n_repetitive":0}
if RESUME:
    last=_newest_ckpt(); ts=os.path.join(last,"trainer_state.pt") if last else None
    if ts and os.path.isfile(ts):
        s=torch.load(ts, map_location="cpu"); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
        for k in state:
            if k in s: state[k]=s[k]
        print(f"↩️ resumed step {state['step']} pass {state['pass_idx']} cursor {state['cursor']}")
correction_buffer=load_buffer()
print(f"buffer {len(correction_buffer)} | ✓{state['n_correct']} ✗{state['n_wrong']} ∅{state['n_unverifiable']} ↻{state['n_repetitive']}")


## 13 · Main OPSD loop — answer-gated, repeat4-gated, resumable, per-sample logging

In [ ]:
import time
model.train()
ema, best_ema = state["ema"], state["best_ema"]
step, pass_idx, cursor = state["step"], state["pass_idx"], state["cursor"]
t_start=time.time(); since=[]
print(f"Training: pass {pass_idx+1}/{cfg.target_passes} | cursor {cursor}/{len(opsd_pool)} | step {step} | "
      f"student {cfg.block_size//cfg.student_steps_per_block}tok/step, teacher {cfg.block_size//cfg.teacher_steps_per_block}tok/step | "
      f"rep4 gate > {cfg.repeat4_max}\n")

def _persist(s_):
    save_ckpt(s_, {"step":s_,"pass_idx":pass_idx,"cursor":cursor,"ema":ema,"best_ema":best_ema,
                   "n_correct":state["n_correct"],"n_wrong":state["n_wrong"],
                   "n_unverifiable":state["n_unverifiable"],"n_repetitive":state["n_repetitive"]})
    save_buffer(correction_buffer)

interrupted=False
try:
    while pass_idx < cfg.target_passes:
        if cfg.max_hours is not None and (time.time()-t_start)/3600 > cfg.max_hours:
            print("⏹ max_hours reached."); break
        model.zero_grad(set_to_none=True); accepted, attempts, acc_loss = 0, 0, 0.0
        while accepted < cfg.grad_accum and attempts < cfg.max_attempts_per_step:
            if cursor >= len(opsd_pool):
                cursor=0; pass_idx+=1; print(f"\n══ pass {pass_idx}/{cfg.target_passes} done ══\n")
                if pass_idx>=cfg.target_passes: break
            ex=opsd_pool[cursor]; cursor+=1; attempts+=1
            snap=snapshot_grads(); t0=time.time()
            r=rollout_and_backward(ex["question"], ex["solution"], ex["gold"]); dt=time.time()-t0
            tag=f"[p{pass_idx+1} {cursor:04d}/{len(opsd_pool)}]"
            box=f"box@{r['tokens_to_box']:>5}/{r['gen_tokens']:>5}tok" if r["tokens_to_box"] else f"no box @{r['gen_tokens']:>5}tok    "
            if r["outcome"]=="correct":
                accepted+=1; acc_loss+=r["loss"]; state["n_correct"]+=1
                print(f"{tag} {dt:6.1f}s | {box} | ✓ correct      | rep4 {r['repeat4']:.3f} ✅ kept | loss {r['loss']:.4f}")
            else:
                restore_grads(snap)
                if r["outcome"]=="repetitive":
                    state["n_repetitive"]+=1
                    print(f"{tag} {dt:6.1f}s | {box} | ↻ repetitive   | rep4 {r['repeat4']:.3f} > {cfg.repeat4_max} ❌ DISCARD")
                elif r["outcome"]=="wrong":
                    state["n_wrong"]+=1
                    correction_buffer.append({"question":ex["question"],"solution":ex["solution"],
                                              "gold":ex["gold"],"wrong":r["pred"],"attempts":0})
                    print(f"{tag} {dt:6.1f}s | {box} | ✗ wrong {str(r['pred'])[:6]}≠{str(ex['gold'])[:6]} | rep4 {r['repeat4']:.3f} ❌→buffer")
                else:
                    state["n_unverifiable"]+=1
                    print(f"{tag} {dt:6.1f}s | {box} | ∅ unverifiable | rep4 {r['repeat4']:.3f} ❌ DISCARD")
        if accepted==0:
            if attempts>0: print(f"— step {step:>4} | 0/{attempts} accepted — no step —\n")
            continue
        torch.nn.utils.clip_grad_norm_(trainable,1.0); opt.step(); sched.step(); step+=1
        avg=acc_loss/accepted; since.append(avg)
        ema=avg if ema is None else cfg.ema_beta*ema+(1-cfg.ema_beta)*avg
        is_best=ema<best_ema
        if is_best: best_ema=ema
        print(f"— step {step:>4} | acc {accepted}/{attempts} | loss {avg:.4f} | ema {ema:.4f} | "
              f"lr {sched.get_last_lr()[0]:.2e} | ✓{state['n_correct']} ✗{state['n_wrong']} "
              f"∅{state['n_unverifiable']} ↻{state['n_repetitive']}{' 🏆' if is_best else ''} —\n")
        if step % cfg.ckpt_every == 0:
            print(f"💾 ckpt step_{step} | mean loss since last {sum(since)/len(since):.4f}"); since=[]; _persist(step)
        if is_best and step>=max(5,cfg.warmup_steps): save_best(step, ema)
except KeyboardInterrupt:
    interrupted=True; print("\n⏹ interrupted — persisting.")
finally:
    _persist(step)
    state.update({"step":step,"pass_idx":pass_idx,"cursor":cursor,"ema":ema,"best_ema":best_ema})
    print(f"\n{'⏹ paused' if interrupted else '✅ stopped'} at step {step}, pass {pass_idx}/{cfg.target_passes}, "
          f"cursor {cursor}/{len(opsd_pool)}, buffer {len(correction_buffer)}")


## 14 · Correction loop — re-feed buffered failures through the same OPSD process

In [ ]:
import time
model.train()
if not correction_buffer:
    print("Buffer empty — nothing to do.")
else:
    print(f"Correction over {len(correction_buffer)} failures\n")
    still, recovered, retired, accb = [], 0, 0, 0
    model.zero_grad(set_to_none=True)
    for k, it in enumerate(correction_buffer):
        snap=snapshot_grads()
        r=rollout_and_backward(it["question"], it["solution"], it["gold"],
                               failure_note=correction_note(it["wrong"], it["gold"]))
        if r["outcome"]=="correct":
            recovered+=1; accb+=1; print(f"[{k+1}] ✓ recovered (loss {r['loss']:.3f})")
            if accb==cfg.grad_accum:
                torch.nn.utils.clip_grad_norm_(trainable,1.0); opt.step(); sched.step()
                model.zero_grad(set_to_none=True); accb=0
        else:
            restore_grads(snap); it["attempts"]=it.get("attempts",0)+1
            if it["attempts"]>=cfg.max_correction_attempts:
                retired+=1; print(f"[{k+1}] ✗ retired ({it['attempts']} tries)")
            else:
                if r["outcome"]=="wrong" and r["pred"] is not None: it["wrong"]=r["pred"]
                still.append(it); print(f"[{k+1}] ✗ kept (try {it['attempts']}, {r['outcome']})")
    if accb>0:
        torch.nn.utils.clip_grad_norm_(trainable,1.0); opt.step(); sched.step(); model.zero_grad(set_to_none=True)
    correction_buffer=still; save_buffer(correction_buffer)
    print(f"\nrecovered {recovered} | retired {retired} | pending {len(correction_buffer)}")
    print("success rate:", f"{100*recovered/max(1,recovered+retired+len(correction_buffer)):.0f}%")


## 15 · Held-out correctness eval (the metric that matters)

In [ ]:
n = sft_eval(eval_holdout, min(cfg.n_eval_holdout, len(eval_holdout)),
              budget=cfg.max_gen_tokens, tag="CURRENT (post-training)")


## 16 · Operating notes (d-OPSD, local)

**Run the null control first.** `cfg.future_reveal_k = 0` in §3, then §11: loss must be ≈0,
because the teacher is then bit-identical to the student. Non-zero loss there means the KL is
being computed on mismatched positions — stop and fix before spending GPU hours. Restore
`future_reveal_k=4` afterwards.

**`future_reveal_k` is the one knob that defines this method.** Too small → teacher ≈ student,
no signal. Too large → teacher sees most of the block and the target stops resembling anything
the student could learn to predict from its own conditional. 2–8 at block-32 is the sensible
sweep; report whichever value is used, and if there is time, run k ∈ {2, 4, 8} on a small pool —
that sweep is a figure on its own.

**Trajectory yield gates everything.** Self-distillation needs usable on-policy rollouts. Earlier
runs on this checkpoint gave 0/28 usable rollouts at temperature 0.9 / budget 768 versus 38/50 at
temperature 0.3 / budget 1536. Watch the ∅-rate; if it is high, fix sampling parameters before
concluding anything about the method.

**Local-run housekeeping.** Checkpoints land in `cfg.drive_root` (`./opsd2_run` by default) and
`RESUME=True` picks up the newest `step_*`. If the kernel dies, just re-run from the top — §6
resumes adapters, §12 restores optimizer/buffer state. Nothing depends on Colab.

**Comparability with OPSD-1.** For the two runs to be comparable, keep these identical across
both notebooks: `model_id`, `block_size`, `student_steps_per_block`, `max_gen_tokens`,
`temperature`, `top_p`, `noise_temp`, `seed`, the eval holdout, and the SFT warmup (or skip SFT
in both). Differences to record explicitly: teacher prompt (privileged vs identical), teacher
schedule (2 tok/step vs 8 tok/step), and `future_reveal_k` (n/a vs its value.)